# Axis 1 — Cache Capacity vs Working Set (Mac / MLX)

**Model:** `allenai/OLMoE-1B-7B-0924-Instruct` (unquantized BF16, MLX format)  
**Architecture:** 16 layers × 64 experts/layer × top-8 routing = 1,024 logical experts  
**Platform:** Apple Silicon, MLX

## True Disk-Backed Expert Cache

This notebook implements a **true K-expert cache** where:

1. Only **non-expert parameters** (~1.5 GB) are loaded into memory
2. Expert weights **stay on disk** in safetensors files
3. On cache miss, **only the requested expert's bytes** are read from disk (~12 MB per expert)
4. At most **K experts per layer** are resident in memory at any time
5. LRU eviction when the cache is full

The complete expert bank (~12.3 GB) is **never** resident in memory.  
Total memory footprint: ~1.8 GB regardless of prompt length.

In [1]:
import gc, time, struct, json, platform
from collections import OrderedDict, Counter
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import mlx.core as mx
import mlx.nn as nn
from mlx_lm.utils import load_config, _get_classes
from mlx_lm.tokenizer_utils import load as load_tokenizer
from mlx_lm.generate import generate_step

assert platform.system() == 'Darwin', 'This notebook requires macOS with Apple Silicon.'
print(f'MLX {mx.__version__} | Python {platform.python_version()}')

MLX 0.32.2 | Python 3.14.6


## Configuration

In [2]:
# ── Paths ──
cwd = Path.cwd()
if cwd.name == 'axis-1-cache-capacity':
    MODEL_DIR = cwd.parent.parent / 'models' / 'olmoe-1b-7b-0924-instruct-mlx'
else:
    MODEL_DIR = cwd / 'models' / 'olmoe-1b-7b-0924-instruct-mlx'

assert MODEL_DIR.exists(), f'Model not found: {MODEL_DIR}'

# ── Experiment parameters ──
CACHE_CAPACITY_K = 16         # K = experts resident per layer
MAX_NEW_TOKENS   = 24        # tokens to generate
PROMPT = 'Explain why loading only the experts selected by a router can reduce memory use.'

print(f'Model dir: {MODEL_DIR}')
print(f'Cache capacity: K = {CACHE_CAPACITY_K} per layer')
print(f'Max new tokens: {MAX_NEW_TOKENS}')

Model dir: /Users/falak/Documents/Falak/College/Sem - 3/DS635/MoE - Inference System/models/olmoe-1b-7b-0924-instruct-mlx
Cache capacity: K = 16 per layer
Max new tokens: 24


## Step 1: Build Expert Disk Index

Parse safetensors file headers to locate each expert's byte range on disk.  
No tensor data is loaded — only the JSON metadata header (a few KB per shard).

Expert tensors have shape `(64, out_dim, in_dim)` in BF16.  
Expert `i` occupies a contiguous byte range: `offset + i * stride` to `offset + (i+1) * stride`.  
Each expert projection is ~4 MB; three projections = ~12 MB per expert.

In [3]:
def parse_safetensors_header(filepath):
    """Read only the JSON header from a safetensors file (no tensor data)."""
    with open(filepath, 'rb') as f:
        header_size = struct.unpack('<Q', f.read(8))[0]
        header_json = json.loads(f.read(header_size))
    return header_size, header_json

DTYPE_BYTES = {'BF16': 2, 'F16': 2, 'F32': 4, 'I32': 4, 'I64': 8}

expert_index = {}                 # (layer, proj_name) -> disk location info
non_expert_keys_by_shard = {}     # shard_file -> [non-expert tensor names]
shard_files = sorted(MODEL_DIR.glob('model*.safetensors'))

for shard_file in shard_files:
    header_size, metadata = parse_safetensors_header(shard_file)
    data_region_start = 8 + header_size
    ne_keys = []

    for tensor_name, info in metadata.items():
        if tensor_name == '__metadata__':
            continue

        is_expert = '.switch_mlp.' in tensor_name and tensor_name.endswith('.weight')

        if is_expert:
            shape = info['shape']
            dtype_str = info['dtype']
            offset_start = info['data_offsets'][0]

            parts = tensor_name.split('.')
            layer_idx = int(parts[2])   # model.layers.{L}.mlp.switch_mlp.{proj}.weight
            proj_name = parts[5]         # gate_proj / up_proj / down_proj

            expert_shape = tuple(shape[1:])  # shape of a single expert's weight
            expert_bytes = 1
            for d in expert_shape:
                expert_bytes *= d
            expert_bytes *= DTYPE_BYTES[dtype_str]

            expert_index[(layer_idx, proj_name)] = {
                'file': str(shard_file),
                'data_start': data_region_start + offset_start,
                'expert_stride': expert_bytes,
                'expert_shape': expert_shape,
                'dtype_str': dtype_str,
            }
        else:
            ne_keys.append(tensor_name)

    non_expert_keys_by_shard[str(shard_file)] = ne_keys

# ── Summary ──
n_expert_entries = len(expert_index)
n_non_expert = sum(len(v) for v in non_expert_keys_by_shard.values())
sample = expert_index[(0, 'gate_proj')]

print(f'Expert index: {n_expert_entries} entries (expect 48 = 16 layers × 3 projections)')
print(f'Non-expert tensors: {n_non_expert}')
print(f'\nSample — layer 0 gate_proj:')
print(f'  Shape per expert: {sample["expert_shape"]}')
print(f'  Bytes per expert: {sample["expert_stride"]:,} ({sample["expert_stride"]/1e6:.1f} MB)')
print(f'  Dtype: {sample["dtype_str"]}')
print(f'  Shard: {Path(sample["file"]).name}')

assert n_expert_entries == 48, f'Expected 48 expert index entries, got {n_expert_entries}'

Expert index: 48 entries (expect 48 = 16 layers × 3 projections)
Non-expert tensors: 147

Sample — layer 0 gate_proj:
  Shape per expert: (1024, 2048)
  Bytes per expert: 4,194,304 (4.2 MB)
  Dtype: BF16
  Shard: model-00001-of-00003.safetensors


## Step 2: Verify Disk Byte-Range Read

Before building the cache, verify that reading raw bytes at computed offsets
produces the **exact same** weight tensor as `mx.load()` + indexing.

In [4]:
# Load full tensor via mx.load to get ground truth
ie = expert_index[(0, 'gate_proj')]
ref_all = mx.load(ie['file'])
ref_e0 = ref_all['model.layers.0.mlp.switch_mlp.gate_proj.weight'][0]
mx.eval(ref_e0)

# Load same expert via raw byte-range read
with open(ie['file'], 'rb') as f:
    f.seek(ie['data_start'])
    raw = f.read(ie['expert_stride'])

# BF16 has no numpy dtype → read as uint16, view as bfloat16 in MLX
disk_e0 = mx.array(
    np.frombuffer(raw, dtype=np.uint16).reshape(ie['expert_shape'])
).view(mx.bfloat16)
mx.eval(disk_e0)

diff = mx.max(mx.abs(ref_e0 - disk_e0)).item()
print(f'Layer 0, gate_proj, expert 0 — mx.load vs byte-range: max_diff = {diff}')
assert diff == 0.0, f'Byte-range read verification FAILED: max_diff = {diff}'
print('✓ Byte-range read produces identical weights')

del ref_all, ref_e0, disk_e0, raw
gc.collect()
mx.clear_cache()

Layer 0, gate_proj, expert 0 — mx.load vs byte-range: max_diff = 0.0
✓ Byte-range read produces identical weights


## Step 3: Load Non-Expert Weights Only

Create the model architecture, then load **only** non-expert parameters:
- Embeddings, layer norms, attention projections, router gate weights
- Approximately 1.5 GB

Expert MLP weights (`switch_mlp.{gate,up,down}_proj.weight`) are **not loaded**.  
They will be replaced by the disk-backed cache before any forward pass.

In [5]:
config = load_config(MODEL_DIR)
tokenizer = load_tokenizer(MODEL_DIR)

model_class, model_args_class = _get_classes(config)
model = model_class(model_args_class.from_dict(config))

rss_before = psutil.Process().memory_info().rss
print('Loading ONLY non-expert weights from safetensors shards...')

for shard_file in shard_files:
    shard_key_set = set(non_expert_keys_by_shard[str(shard_file)])
    all_weights = mx.load(str(shard_file))
    non_expert_w = {k: v for k, v in all_weights.items() if k in shard_key_set}
    model.load_weights(list(non_expert_w.items()), strict=False)
    del all_weights, non_expert_w
    gc.collect()

rss_after = psutil.Process().memory_info().rss
print(f'Done. RSS: {rss_before/1e9:.2f} → {rss_after/1e9:.2f} GB '
      f'(+{(rss_after-rss_before)/1e9:.2f} GB)')

# Sanity: router weights are loaded correctly
gate_w = model.model.layers[0].mlp.gate['weight']
mx.eval(gate_w)
print(f'Router gate[0] shape: {gate_w.shape}, sum: {mx.sum(gate_w).item():.4f} (non-zero → loaded)')

Loading ONLY non-expert weights from safetensors shards...
Done. RSS: 0.88 → 0.61 GB (+-0.27 GB)
Router gate[0] shape: (64, 2048), sum: -15.0625 (non-zero → loaded)


## Step 4: Disk-Backed Expert Cache

Each layer gets a `DiskBackedExpertCache` with capacity K.  
On cache miss, it reads a single expert's 3 projections (~12 MB) from disk  
via `seek()` + `read()` at the pre-computed byte offset. No full tensor materialization.

The cache replaces `switch_mlp` on each `OlmoeSparseMoeBlock`, so the router
calls `cache(x_flat, indices)` instead of the original `SwitchGLU`.

Per-expert computation: `out = silu(x @ gate.T) * (x @ up.T) @ down.T`  
(validated as numerically identical to `gather_mm`-based `SwitchGLU`).

In [6]:
class DiskBackedExpertCache:
    """Per-layer LRU expert cache.
    
    Expert weights stay on disk in safetensors files.
    Only `capacity` experts are resident in memory at any time.
    On a miss, reads the expert's byte range from disk (~12 MB).
    """

    def __init__(self, layer_idx, expert_index, capacity=4):
        self.layer_idx = layer_idx
        self.capacity = max(1, int(capacity))
        self.cache = OrderedDict()   # expert_id -> (gate_w, up_w, down_w)
        
        # Counters
        self.hits = 0
        self.misses = 0
        self.evictions = 0
        self.bytes_loaded = 0
        self.load_time_ms = 0.0
        self.accesses = Counter()

        # Pre-resolved disk locations for this layer's 3 projections
        self.gate_idx = expert_index[(layer_idx, 'gate_proj')]
        self.up_idx   = expert_index[(layer_idx, 'up_proj')]
        self.down_idx = expert_index[(layer_idx, 'down_proj')]

    def _read_expert(self, ie, eid):
        """Read one expert's weight from a safetensors file via byte-range seek."""
        offset = ie['data_start'] + eid * ie['expert_stride']
        with open(ie['file'], 'rb') as f:
            f.seek(offset)
            raw = f.read(ie['expert_stride'])
        arr = np.frombuffer(raw, dtype=np.uint16).reshape(ie['expert_shape'])
        return mx.array(arr).view(mx.bfloat16)

    def get(self, eid):
        """Get expert weights, loading from disk on miss. Returns (gate_w, up_w, down_w)."""
        eid = int(eid)
        self.accesses[eid] += 1

        if eid in self.cache:
            self.hits += 1
            w = self.cache.pop(eid)      # move to end (most recently used)
            self.cache[eid] = w
            return w

        # Cache miss — load from disk
        self.misses += 1
        t0 = time.perf_counter()
        gw = self._read_expert(self.gate_idx, eid)
        uw = self._read_expert(self.up_idx, eid)
        dw = self._read_expert(self.down_idx, eid)
        self.load_time_ms += (time.perf_counter() - t0) * 1000
        self.bytes_loaded += sum(
            ie['expert_stride'] for ie in [self.gate_idx, self.up_idx, self.down_idx]
        )

        w = (gw, uw, dw)
        self.cache[eid] = w

        # Evict LRU if over capacity
        if len(self.cache) > self.capacity:
            _, evicted = self.cache.popitem(last=False)
            del evicted
            self.evictions += 1

        return w

    def __call__(self, x_flat, indices):
        """Drop-in replacement for SwitchGLU.__call__.
        
        Args:
            x_flat: (N, D) input tensor
            indices: (N, top_k) expert indices from router
        Returns:
            (N, top_k, D) output tensor
        """
        N, k = indices.shape
        il = indices.tolist()

        if N == 1:  # ── Decode: single token ──
            y = []
            for eid in il[0]:
                gw, uw, dw = self.get(eid)
                g = x_flat @ gw.T
                u = x_flat @ uw.T
                h = nn.silu(g) * u
                y.append(h @ dw.T)
            return mx.stack(y, axis=1)

        # ── Prefill: batch by unique expert for efficiency ──
        expert_map = {}  # eid -> [(token_idx, slot_idx), ...]
        for t in range(N):
            for p in range(k):
                eid = il[t][p]
                expert_map.setdefault(eid, []).append((t, p))

        y = [[None] * k for _ in range(N)]
        for eid, positions in expert_map.items():
            gw, uw, dw = self.get(eid)
            token_indices = [t for t, p in positions]
            xt = x_flat[token_indices]         # (M, D)
            g = xt @ gw.T                      # (M, intermediate)
            u = xt @ uw.T
            h = nn.silu(g) * u
            out = h @ dw.T                     # (M, D)
            for i, (t, p) in enumerate(positions):
                y[t][p] = out[i]

        return mx.stack([mx.stack(row) for row in y])

    def summary(self):
        """Return a dict of cache statistics for this layer."""
        total = self.hits + self.misses
        return {
            'layer': self.layer_idx,
            'K': self.capacity,
            'resident': len(self.cache),
            'accesses': total,
            'hits': self.hits,
            'misses': self.misses,
            'hit_rate': f'{self.hits/total:.1%}' if total else 'N/A',
            'evictions': self.evictions,
            'MB_loaded': round(self.bytes_loaded / 1e6, 1),
            'io_ms': round(self.load_time_ms, 1),
            'unique_experts': len(self.accesses),
        }

print('DiskBackedExpertCache defined.')

DiskBackedExpertCache defined.


## Step 5: Install Cache on All Layers

Replace `switch_mlp` on each `OlmoeSparseMoeBlock` with a `DiskBackedExpertCache`.  
The original uninitialized expert weight tensors are dropped and garbage-collected.  
After this cell, the model's memory footprint should be ~1.5 GB (non-expert params only).

In [7]:
layer_caches = {}
for i in range(len(model.model.layers)):
    cache = DiskBackedExpertCache(i, expert_index, capacity=CACHE_CAPACITY_K)
    layer_caches[i] = cache
    model.model.layers[i].mlp.switch_mlp = cache

gc.collect()
mx.clear_cache()

metal_mem = mx.get_active_memory() / 1e9
rss_mem = psutil.Process().memory_info().rss / 1e9

print(f'Installed DiskBackedExpertCache on {len(layer_caches)} layers (K={CACHE_CAPACITY_K})')
print(f'Active Metal memory: {metal_mem:.2f} GB')
print(f'RSS: {rss_mem:.2f} GB')
print(f'\n→ Expert bank is NOT resident (expected ~1.5-2 GB, not ~14 GB)')

Installed DiskBackedExpertCache on 16 layers (K=16)
Active Metal memory: 0.00 GB
RSS: 0.61 GB

→ Expert bank is NOT resident (expected ~1.5-2 GB, not ~14 GB)


## Step 6: Generate with Cache

Run end-to-end generation using `generate_step` (low-level API, avoids `wired_limit`).  
The cache will load experts from disk on miss during both prefill and decode.

In [8]:
# Tokenize prompt
messages = [{'role': 'user', 'content': PROMPT}]
prompt_str = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False
)
prompt_tokens = mx.array(tokenizer.encode(prompt_str))

print(f'Prompt: "{PROMPT}"')
print(f'Prompt tokens: {len(prompt_tokens)}')
print(f'Max new tokens: {MAX_NEW_TOKENS}')
print(f'\nGenerating...')

# ── Generation loop ──
t_start = time.perf_counter()
generated_tokens = []

for token, logprobs in generate_step(prompt_tokens, model, max_tokens=MAX_NEW_TOKENS):
    generated_tokens.append(token)

t_total = time.perf_counter() - t_start
generated_text = tokenizer.decode(generated_tokens)

print(f'\n=== GENERATION RESULT ===')
print(f'Tokens: {len(generated_tokens)}')
print(f'Wall time: {t_total*1000:.0f} ms')
print(f'Tokens/sec: {len(generated_tokens)/t_total:.2f}')
print(f'Response: {generated_text}')

print(f'\n=== MEMORY ===')
print(f'Active Metal memory: {mx.get_active_memory()/1e9:.2f} GB')
print(f'Peak Metal memory: {mx.get_peak_memory()/1e9:.2f} GB')
print(f'RSS: {psutil.Process().memory_info().rss/1e9:.2f} GB')

Prompt: "Explain why loading only the experts selected by a router can reduce memory use."
Prompt tokens: 29
Max new tokens: 24

Generating...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPTNeoXTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



=== GENERATION RESULT ===
Tokens: 24
Wall time: 13372 ms
Tokens/sec: 1.79
Response: Loading only the experts selected by a router can reduce memory use for several reasons:

1. Reduced Model Size

=== MEMORY ===
Active Metal memory: 4.18 GB
Peak Metal memory: 4.43 GB
RSS: 3.82 GB


## Step 7: Cache Statistics

In [9]:
# ── Per-layer statistics ──
stats = pd.DataFrame([c.summary() for c in layer_caches.values()])
print('Per-layer cache statistics:')
print(stats.to_string(index=False))

# ── Global summary ──
total_accesses = sum(c.hits + c.misses for c in layer_caches.values())
total_hits     = sum(c.hits for c in layer_caches.values())
total_misses   = sum(c.misses for c in layer_caches.values())
total_evictions = sum(c.evictions for c in layer_caches.values())
total_io_ms    = sum(c.load_time_ms for c in layer_caches.values())
total_bytes    = sum(c.bytes_loaded for c in layer_caches.values())

print(f'\n{"="*50}')
print(f'GLOBAL CACHE SUMMARY  (K = {CACHE_CAPACITY_K})')
print(f'{"="*50}')
print(f'Total accesses:     {total_accesses}')
print(f'Total hits:         {total_hits} ({total_hits/total_accesses:.1%})')
print(f'Total misses:       {total_misses} ({total_misses/total_accesses:.1%})')
print(f'Total evictions:    {total_evictions}')
print(f'Data loaded:        {total_bytes/1e6:.0f} MB')
print(f'Disk I/O time:      {total_io_ms:.0f} ms')
print(f'Generation time:    {t_total*1000:.0f} ms')
print(f'I/O fraction:       {total_io_ms/(t_total*1000):.1%}')
print(f'Unique experts seen: {sum(len(c.accesses) for c in layer_caches.values())} '
      f'across all layers')

Per-layer cache statistics:
 layer  K  resident  accesses  hits  misses hit_rate  evictions  MB_loaded  io_ms  unique_experts
     0 16        16       256    89     167    34.8%        151     2101.3  710.1              59
     1 16        16       254    67     187    26.4%        171     2353.0  793.3              57
     2 16        16       254   104     150    40.9%        134     1887.4  695.2              59
     3 16        16       251    95     156    37.8%        140     1962.9  709.3              56
     4 16        16       252   109     143    43.3%        127     1799.4  673.5              57
     5 16        16       249   122     127    49.0%        111     1598.0  614.7              52
     6 16        16       245   120     125    49.0%        109     1572.9  678.2              50
     7 16        16       253   108     145    42.7%        129     1824.5  675.1              56
     8 16        16       251   100     151    39.8%        135     1900.0  712.2         

## Step 8: Validation

Verify that this is a **true** K-expert cache — not a disguised full-bank-resident system.

In [10]:
metal_gb = mx.get_active_memory() / 1e9
max_resident = max(len(c.cache) for c in layer_caches.values())
resident_expert_mb = max_resident * 3 * 4.0  # 3 projections × ~4 MB each

print(f'=== VALIDATION ===')
print(f'Cache capacity:           K = {CACHE_CAPACITY_K} per layer')
print(f'Max resident per layer:   {max_resident} experts')
print(f'Resident expert memory:   ~{resident_expert_mb:.0f} MB per layer')
print(f'Active Metal memory:      {metal_gb:.2f} GB')
print(f'Tokens generated:         {len(generated_tokens)}')
print()

# ── Assertions ──
assert metal_gb < 4.0, (
    f'Active Metal memory = {metal_gb:.2f} GB — '
    f'expert bank may still be resident! Expected < 4 GB.'
)
print('✓ Active Metal memory < 4 GB — expert bank is NOT resident')

assert max_resident <= CACHE_CAPACITY_K, (
    f'Max resident experts = {max_resident}, exceeds K = {CACHE_CAPACITY_K}'
)
print(f'✓ Max resident experts = {max_resident} ≤ K = {CACHE_CAPACITY_K}')

assert len(generated_tokens) == MAX_NEW_TOKENS, (
    f'Expected {MAX_NEW_TOKENS} tokens, got {len(generated_tokens)}'
)
print(f'✓ Generated {len(generated_tokens)} tokens successfully')

print(f'\n✅ TRUE K-EXPERT DISK-BACKED CACHE VERIFIED')

=== VALIDATION ===
Cache capacity:           K = 16 per layer
Max resident per layer:   16 experts
Resident expert memory:   ~192 MB per layer
Active Metal memory:      4.17 GB
Tokens generated:         24



AssertionError: Active Metal memory = 4.17 GB — expert bank may still be resident! Expected < 4 GB.